Your Name and PRN:
- Name: Aaryan Bairagi
- PRN : 8001
- Date: 10/11/2026

# Image Processing with Neural Network

## Assignment: A04
### Take Neural Network with :
- Multiple hidden layers
- Activation function of your choice


### Assignment
- **Q1**: Is this model more accurate compared to previous model?
- **Q2**: Prepare table by changing number of neurons in hidden layer, learning rate and observe change in results. Also comment on your results.

|#|Dimension of hidden layer|Learning rate|Training Accuracy|Test Accuracy|Comment|
|:-:|:-:|:-:|:-:|:-:|:--|
|1|5-5-4-3-2|0.1|0.97|0.96|Base case||1|4|0.1|0.97|0.96|Base case|
|2|10-5-4-3-2|1|???|???|???|
|...|...|...|...|...|...|
|n|...|...|...|...|...|

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)

Using device: cuda


100%|██████████| 9.91M/9.91M [00:00<00:00, 18.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 469kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.52MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.3MB/s]


In [ ]:
class FlexibleNN(nn.Module):
    def __init__(self, input_size, hidden_dims, num_classes):
        super(FlexibleNN, self).__init__()
        layers = []

        prev_dim = input_size

        for dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, dim))
            layers.append(nn.ReLU())
            prev_dim = dim

        layers.append(nn.Linear(prev_dim, num_classes))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.network(x)

In [ ]:
def evaluate_accuracy(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

def train_and_evaluate(hidden_dims, lr, epochs=5):
    input_size = 28 * 28
    num_classes = 10

    model = FlexibleNN(input_size, hidden_dims, num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)

    print(f"Training Model with Hidden Layers: {hidden_dims} | Learning Rate: {lr}")

    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    train_acc = evaluate_accuracy(model, train_loader)
    test_acc = evaluate_accuracy(model, test_loader)

    print(f"-> Training Accuracy: {train_acc:.4f} | Test Accuracy: {test_acc:.4f}\n")
    return train_acc, test_acc

In [ ]:

hidden_1 = [5, 5, 4, 3, 2]
lr_1 = 0.1
acc_train_1, acc_test_1 = train_and_evaluate(hidden_1, lr_1, epochs=5)

hidden_2 = [10, 5, 4, 3, 2]
lr_2 = 1.0
acc_train_2, acc_test_2 = train_and_evaluate(hidden_2, lr_2, epochs=5)


hidden_3 = [128, 64]
lr_3 = 0.05
acc_train_3, acc_test_3 = train_and_evaluate(hidden_3, lr_3, epochs=5)

Training Model with Hidden Layers: [5, 5, 4, 3, 2] | Learning Rate: 0.1
-> Training Accuracy: 0.1124 | Test Accuracy: 0.1135

Training Model with Hidden Layers: [10, 5, 4, 3, 2] | Learning Rate: 1.0
-> Training Accuracy: 0.1124 | Test Accuracy: 0.1135

Training Model with Hidden Layers: [128, 64] | Learning Rate: 0.05
-> Training Accuracy: 0.9823 | Test Accuracy: 0.9704



In [ ]:
configs = [
    ([5, 5, 4, 3, 2], 0.1, "Base Bottleneck"),
    ([10, 5, 4, 3, 2], 1.0, "High LR Bottleneck"),
    ([128, 64], 0.05, "Standard Wide"),
    ([64, 32], 0.1, "Medium Wide")
]

results = []
for hidden_dims, lr, comment in configs:
    try:
        train_acc, test_acc = train_and_evaluate(hidden_dims, lr, epochs=5)
        results.append({
            "Hidden Layers": "-".join(map(str, hidden_dims)),
            "Learning Rate": lr,
            "Training Accuracy": f"{train_acc:.4f}",
            "Test Accuracy": f"{test_acc:.4f}",
            "Comment": comment
        })
    except Exception as e:
        print(f"Error with configuration {hidden_dims}: {e}")

import pandas as pd
df_results = pd.DataFrame(results)
display(df_results)

Training Model with Hidden Layers: [5, 5, 4, 3, 2] | Learning Rate: 0.1
-> Training Accuracy: 0.6684 | Test Accuracy: 0.6651

Training Model with Hidden Layers: [10, 5, 4, 3, 2] | Learning Rate: 1.0
-> Training Accuracy: 0.1044 | Test Accuracy: 0.1028

Training Model with Hidden Layers: [128, 64] | Learning Rate: 0.05
-> Training Accuracy: 0.9797 | Test Accuracy: 0.9691

Training Model with Hidden Layers: [64, 32] | Learning Rate: 0.1
-> Training Accuracy: 0.9842 | Test Accuracy: 0.9724



,Hidden Layers,Learning Rate,Training Accuracy,Test Accuracy,Comment
0,5-5-4-3-2,0.10,0.6684,0.6651,Base Bottleneck
1,10-5-4-3-2,1.00,0.1044,0.1028,High LR Bottleneck
2,128-64,0.05,0.9797,0.9691,Standard Wide
3,64-32,0.10,0.9842,0.9724,Medium Wide
